# AN-RA iterate500 T4 Training
Canonical bootstrap, preflight, 500M-class frontier training, resume, and ThirdEye evaluation. The trainer restores from MyDrive checkpoints first, then visible shared Drive locations / Shared-with-me when available. Secrets must be supplied through Colab secrets/environment variables, never notebook cells.

In [ ]:
from google.colab import drive
from pathlib import Path

def mount_drive_or_stop():
    for force in (False, True):
        try:
            drive.mount('/content/drive', force_remount=force)
            if Path('/content/drive/MyDrive').exists():
                print('[Drive] mounted at /content/drive')
                return
        except Exception as exc:
            print(f'[Drive] mount attempt force_remount={force} failed: {type(exc).__name__}: {exc}')
    raise RuntimeError('Google Drive auth failed before training. In Colab: Runtime -> Disconnect and delete runtime, reload the notebook, sign into the correct Google account, then run again. Training needs Drive for checkpoint resume/save.')

mount_drive_or_stop()

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Wrong Colab runtime. Choose Runtime -> Change runtime type -> T4 GPU. TPU v5e/CPU should use the TPU notebook.')
print('gpu:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
REPO = '/content/An-Ra-the-new-AGI'
BRANCH = 'iterate500'
![ -d "$REPO/.git" ] || git clone --branch "$BRANCH" --single-branch https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git "$REPO"
%cd $REPO
!git fetch origin "$BRANCH"
!git checkout "$BRANCH"
!git pull --ff-only origin "$BRANCH"
import os
from pathlib import Path
PIP_CACHE = Path('/content/.cache/pip')
PIP_CACHE.mkdir(parents=True, exist_ok=True)
os.environ['PIP_CACHE_DIR'] = str(PIP_CACHE)
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
os.environ['PIP_NO_INPUT'] = '1'
!python scripts/colab_bootstrap.py --repo "$REPO" --drive-root /content/drive/MyDrive/AnRa --install --install-thirdeye --model-size frontier

In [ ]:
%cd /content/An-Ra-the-new-AGI
import os
from pathlib import Path
from training.shared_checkpoint import restore_shared_checkpoint

# This branch continues the existing 500M experiment. Set False only for a truly new model.
REQUIRE_RESUME = True
CHECKPOINT = Path('anra_frontier_500m.pt')
os.environ['ANRA_SHARED_DRIVE_API'] = '1'
# This run has one Drive checkpoint only: the owner master in My Drive or its editable Shared with me view.
# Training stops rather than creating a private or versioned checkpoint copy.
os.environ['ANRA_REQUIRE_SHARED_MASTER'] = '1'
os.environ['ANRA_REQUIRE_RESUME'] = '1' if REQUIRE_RESUME else '0'
source = restore_shared_checkpoint(CHECKPOINT)
if CHECKPOINT.exists():
    print(f'[Resume Check] READY: {CHECKPOINT} ({CHECKPOINT.stat().st_size / 1024**3:.2f} GB), source={source}')
elif REQUIRE_RESUME:
    raise RuntimeError('Resume checkpoint was not found. Training is intentionally stopped so it cannot restart at step 1. Confirm the same Google account owns or can access anra_frontier_500m.pt, then rerun this cell.')
else:
    print('[Resume Check] No checkpoint found. Fresh training is explicitly allowed.')

In [ ]:
%cd /content/An-Ra-the-new-AGI
# Keep this practical Drive-sized corpus profile unchanged across resumes.
DATA_PROFILE = 't4-cached'
FORCE_DATA_REBUILD = False
data_args = '--force-rebuild' if FORCE_DATA_REBUILD else ''
!python scripts/colab_prepare_data.py --repo /content/An-Ra-the-new-AGI --profile $DATA_PROFILE --drive-root /content/drive/MyDrive/AnRa $data_args
!python -m data.causal_corpus
!python scripts/show_thirdeye_summary.py --profile quick --without-model

In [ ]:
import os
# Saved into every checkpoint. A future mismatch is rejected before training.
os.environ['ANRA_DATA_PROFILE'] = DATA_PROFILE
os.environ['ANRA_TRAINING_DATA_LAYOUT'] = 'bucket_packed_v1'
os.environ.setdefault('ANRA_CHECKPOINT_EVERY_MIN', '45')
os.environ.setdefault('ANRA_THIRDEYE_INTELLIGENCE', '1')
os.environ.setdefault('ANRA_THIRDEYE_SAMPLE_EVERY', '50')
SESSION_MINUTES = 180
print('LOSS VIEW: this cell is pure training. Watch the step/loss/best lines here.')
!python scripts/build_brain.py --data_path training_data/anra_training.txt --checkpoint_path anra_frontier_500m.pt --model-size frontier --batch_size 1 --optimizer adafactor --max_minutes $SESSION_MINUTES

In [ ]:
%cd /content/An-Ra-the-new-AGI
print('THIRD EYE VIEW: evidence dashboard after training. This is separate from loss.')
!python scripts/show_thirdeye_summary.py --profile quick --without-model